## Read currency JSON

In [0]:
from pyspark.sql import functions as F

currency_path = (
    "abfss://reference@stnyctaxide2026nandi.dfs.core.windows.net/"
    "currency/usd_inr/exchange_rates.json"
)

df_currency = (
    spark.read
    .option("multiLine", "true")
    .json(currency_path)
)

df_currency.printSchema()
display(df_currency)

## Standardize the currency reference

In [0]:
df_currency_clean = (
    df_currency
    .select(
        F.to_date("date").alias("rate_date"),
        F.upper(F.trim("base")).alias("base_currency"),
        F.upper(F.trim("quote")).alias("target_currency"),
        F.col("rate").cast("double").alias("exchange_rate")
    )
    .withColumn("source", F.lit("Frankfurter API"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

df_currency_clean.printSchema()

display(
    df_currency_clean
    .orderBy("rate_date")
)

In [0]:
df_currency_clean.agg(
    F.count("*").alias("rate_count"),
    F.min("rate_date").alias("min_date"),
    F.max("rate_date").alias("max_date"),
    F.min("exchange_rate").alias("min_rate"),
    F.max("exchange_rate").alias("max_rate")
).show()

## Persist it AND write 

In [0]:
currency_gold_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "reference/currency_usd_inr"
)

(
    df_currency_clean
    .write
    .format("delta")
    .mode("overwrite")
    .save(currency_gold_path)
)

print("Currency reference Delta dataset written successfully.")

## Read the persisted datasets

In [0]:
daily_mart_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/daily_taxi_metrics"
)

currency_gold_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "reference/currency_usd_inr"
)

df_daily_mart = (
    spark.read
    .format("delta")
    .load(daily_mart_path)
)

df_currency_gold = (
    spark.read
    .format("delta")
    .load(currency_gold_path)
)

print(f"Daily mart rows: {df_daily_mart.count():,}")
print(f"Currency reference rows: {df_currency_gold.count():,}")

### Join and Enrich

In [0]:
df_daily_currency = (
    df_daily_mart
    .join(
        df_currency_gold.select(
            F.col("rate_date").alias("pickup_date"),
            "base_currency",
            "target_currency",
            "exchange_rate"
        ),
        "pickup_date",
        "left"
    )
    .withColumn(
        "exchange_rate",
        F.when(
            F.col("pickup_date") == F.lit("2025-12-31"),
            F.lit(89.78)
        ).otherwise(F.col("exchange_rate"))
    )
    .withColumn(
        "total_revenue_inr",
        F.round(F.col("total_revenue") * F.col("exchange_rate"), 2)
    )
    .withColumn(
        "avg_trip_amount_inr",
        F.round(F.col("avg_trip_amount") * F.col("exchange_rate"), 2)
    )
    .withColumn(
        "avg_fare_amount_inr",
        F.round(F.col("avg_fare_amount") * F.col("exchange_rate"), 2)
    )
    .withColumn(
        "avg_tip_amount_inr",
        F.round(F.col("avg_tip_amount") * F.col("exchange_rate"), 2)
    )
)

In [0]:
# Validatye

df_daily_currency.select(
    "pickup_date",
    "total_trips",
    "total_revenue",
    "exchange_rate",
    "total_revenue_inr",
    "avg_trip_amount_inr"
).orderBy("pickup_date").show(20, False)

In [0]:
df_daily_currency.agg(
    F.count("*").alias("days"),
    F.sum(
        F.when(F.col("exchange_rate").isNull(), 1).otherwise(0)
    ).alias("missing_rates"),
    F.min("exchange_rate").alias("min_rate"),
    F.max("exchange_rate").alias("max_rate")
).show()

## Peresisted Enrich Gold Mart

In [0]:
currency_enriched_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/daily_taxi_metrics_usd_inr"
)

(
    df_daily_currency
    .write
    .format("delta")
    .mode("overwrite")
    .save(currency_enriched_path)
)

print("Currency-enriched daily mart written successfully.")

In [0]:
df_currency_mart = (
    spark.read
    .format("delta")
    .load(currency_enriched_path)
)

print(f"Rows: {df_currency_mart.count():,}")

df_currency_mart.agg(
    F.count("*").alias("days"),
    F.sum(
        F.when(F.col("exchange_rate").isNull(), 1).otherwise(0)
    ).alias("missing_rates")
).show()